# Day 2 homework
This excercise streams a generated brouchere in Spanish from the website the user provides using ollama.
It uses Gradio to show an UI to enter the website URL to be scrapped.

In [ ]:
# imports
import os
import json
import gradio as gr
from IPython.display import Markdown, display, update_display
from modified_scrapper import fetch_website_links, fetch_website_contents
from openai import OpenAI


In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
MODEL = "llama3.2"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [ ]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [ ]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links


In [ ]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result


In [ ]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [ ]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt


In [ ]:
def stream_brochure(company_name, url, ollama_model):
    response = ollama.chat.completions.create(
        model=ollama_model,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ]
    )
    result = response.choices[0].message.content

    stream = ollama.chat.completions.create(
        model=ollama_model,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": "Translate the following marketing brochure to Spanish: " + result}
          ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)


In [ ]:
company_input = gr.Textbox(label="Your company:", info="Enter the company name")
url_input = gr.Textbox(label="Your URL:", info="Enter the company URL")
model_selector = gr.Dropdown(["llama3.1:8b", "llama3.2"], label="Select model", value="llama3.1:8b")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_brochure,
    title="Generate Company Brochure and Translate to Spanish", 
    inputs=[company_input, url_input, model_selector], 
    outputs=[message_output], 
    examples=[
            ["Nebula", "https://www.nebula.io/", "llama3.2"]
        ], 
    flagging_mode="never"
    )
view.launch()